In [1]:
"""
Shared evaluation logic for MSE watermark test scripts.
Use Encoder/Decoder from model.conv2_mel_modules and run clean + robust evals.
"""
import os
import csv
import random
from pathlib import Path
from collections import defaultdict

import torch
import yaml
from torch.nn.functional import mse_loss
from torch.utils.data import DataLoader
from rich.progress import track

from model.conv2_mel_modules import Encoder, Decoder
from dataset.data import WavDataset, collate_fn
# from watermarking_model.distortions.dl import distortion


def set_random_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def generate_random_msg(batch_size, msg_length, device):
    return (torch.randint(0, 2, (batch_size, 1, msg_length), device=device).float() * 2) - 1


def load_configs(script_dir=None):
    if script_dir is None:
        script_dir = os.path.dirname(os.path.abspath(__file__))
    config_dir = os.path.join(script_dir, "config")
    process_config = yaml.load(
        open(os.path.join(config_dir, "process.yaml"), "r"), Loader=yaml.FullLoader
    )
    model_config = yaml.load(
        open(os.path.join(config_dir, "model.yaml"), "r"), Loader=yaml.FullLoader
    )
    train_config = yaml.load(
        open(os.path.join(config_dir, "train.yaml"), "r"), Loader=yaml.FullLoader
    )
    return process_config, model_config, train_config


def load_encoder_decoder(process_config, model_config, train_config, checkpoint_path, device):
    msg_length = train_config["watermark"]["length"]
    encoder = Encoder(
        process_config, model_config, train_config, msg_length
    ).to(device)
    decoder = Decoder(
        process_config, model_config, train_config, msg_length
    ).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    encoder.load_state_dict(checkpoint["encoder"])
    decoder.load_state_dict(checkpoint["decoder"])
    return encoder, decoder


def to_B1T(x):
    if x.dim() == 1:
        x = x.unsqueeze(0)
    if x.dim() == 2:
        x = x.unsqueeze(1)
    return x


def to_BT(x):
    if x.dim() == 3 and x.size(1) == 1:
        x = x.squeeze(1)
    return x


# Attack names and wave-only attack IDs (skip spectrogram-returning attacks)
ATTACK_NAMES = {
    0: "none",
    1: "crop(rand small)",
    2: "crop(10%)",
    3: "resample(noop)",
    4: "crop_front",
    5: "crop_middle",
    6: "crop_back",
    7: "resample->22.05k->back",
    8: "resample->8k->back",
    9: "gaussian_noise",
    10: "amp_scale",
    11: "mp3",
    12: "recount(8bit)",
    13: "median_filter",
    14: "low_pass_2k",
    15: "high_pass_500",
    16: "modify_mel",
    19: "crop_mel_wave_front",
    20: "crop_mel_wave_back",
    22: "crop_mel_wave_position(1..10)",
    24: "crop_mel_wave_position_5bins(1..20)",
    26: "crop_mel_wave_position_20bins(1..5)",
    27: "benign_reencode",
    28: "benign_noise_suppression",
    29: "benign_compression",
    30: "benign_phone_disortion",
}

WAVE_ATTACKS = [9, 27, 28, 29, 30]

RATIO_BY_ATTACK = {
    4: 10,
    5: 10,
    6: 10,
    9: 20,
    10: 50,
    11: 64,
    13: 3,
    16: 50,
    19: 30,
    20: 30,
    22: 5,
    24: 5,
    26: 2,
}


def run_clean_eval(encoder, decoder, dev_audios_loader, train_config, device):
    """Run clean (no distortion) eval; return test_avg_acc, test_avg_snr, count."""
    test_avg_acc = [0.0, 0.0]
    test_avg_snr = 0.0
    count = 0
    msg_length = train_config["watermark"]["length"]

    with torch.inference_mode():
        encoder.eval()
        decoder.eval()
        for sample in track(dev_audios_loader):
            wav_matrix = sample["matrix"].to(device)
            msg = generate_random_msg(wav_matrix.size(0), msg_length, device)
            out = encoder(wav_matrix, msg, 1)
            if out[0] is None:
                continue
            watermark = out[0]
            y_wm = wav_matrix + watermark
            decoded = decoder(y_wm, 1)
            decoder_acc = [
                ((decoded[0] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
                ((decoded[1] >= 0).eq(msg >= 0).sum().float() / msg.numel()).item(),
            ]
            zero_tensor = torch.zeros(wav_matrix.shape, device=wav_matrix.device)
            snr = 10 * torch.log10(
                mse_loss(wav_matrix.detach(), zero_tensor)
                / (mse_loss(wav_matrix.detach(), y_wm.detach()) + 1e-12)
            )
            test_avg_snr += snr.item()
            test_avg_acc[0] += decoder_acc[0]
            test_avg_acc[1] += decoder_acc[1]
            count += 1

    if count == 0:
        return test_avg_acc, test_avg_snr, 0
    test_avg_acc[0] /= count
    test_avg_acc[1] /= count
    test_avg_snr /= count
    return test_avg_acc, test_avg_snr, count


# def run_robust_eval(encoder, decoder, dev_audios_loader, process_config, train_config, device):
#     """Run robust eval over WAVE_ATTACKS; return sums dict for CSV."""
#     eps = 1e-12
#     attacker = distortion(process_config).to(device)
#     sums = defaultdict(
#         lambda: {
#             "acc_batch_sum": 0.0,
#             "snr_total_sum": 0.0,
#             "snr_vs_wm_sum": 0.0,
#             "batch_count": 0,
#         }
#     )
#     msg_length = train_config["watermark"]["length"]
#
#     with torch.inference_mode():
#         encoder.eval()
#         decoder.eval()
#         for sample in track(dev_audios_loader):
#             wav = sample["matrix"].to(device)
#             wav_BT = to_BT(wav)
#             wav_B1T = to_B1T(wav_BT)
#             B, T_ref = wav_BT.size(0), wav_BT.size(1)
#             msg = generate_random_msg(B, msg_length, device)
#             tgt = msg >= 0
#
#             out = encoder(wav_BT, msg, 1)
#             if out[0] is None:
#                 continue
#             watermark = out[0]
#             y_wm_BT = wav_BT + watermark
#             y_wm_B1T = to_B1T(y_wm_BT)
#
#             for aid in WAVE_ATTACKS:
#                 ratio = RATIO_BY_ATTACK.get(aid, 10)
#                 y_dist_B1T = attacker(y_wm_B1T, attack_choice=aid, ratio=ratio)
#                 y_dist_BT = to_BT(y_dist_B1T)
#                 decoded = decoder(y_dist_BT, 1)
#                 batch_acc = (decoded[0] >= 0).eq(tgt).float().mean().item()
#
#                 T_hat = y_dist_BT.size(1)
#                 T = min(T_ref, T_hat)
#                 wav_cut = wav_BT[:, :T]
#                 y_wm_cut = y_wm_BT[:, :T]
#                 y_dist_cut = y_dist_BT[:, :T]
#                 sig_pow = (wav_cut ** 2).mean(dim=1)
#                 noise_pow = ((y_dist_cut - wav_cut) ** 2).mean(dim=1)
#                 snr_total = 10.0 * torch.log10((sig_pow + eps) / (noise_pow + eps))
#                 wm_pow = (y_wm_cut ** 2).mean(dim=1)
#                 noise_wm_p = ((y_dist_cut - y_wm_cut) ** 2).mean(dim=1)
#                 snr_vs_wm = 10.0 * torch.log10((wm_pow + eps) / (noise_wm_p + eps))
#
#                 sums[aid]["acc_batch_sum"] += batch_acc
#                 sums[aid]["snr_total_sum"] += snr_total.mean().item()
#                 sums[aid]["snr_vs_wm_sum"] += snr_vs_wm.mean().item()
#                 sums[aid]["batch_count"] += 1
#
#     return sums


def write_clean_csv(results_dir, csv_path, dataset, test_avg_acc, test_avg_snr, num_samples):
    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)
    csv_path = results_dir / csv_path
    file_exists = csv_path.exists()
    snr_val = float(
        test_avg_snr.item() if torch.is_tensor(test_avg_snr) else test_avg_snr
    )
    with open(csv_path, mode="a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(
                ["dataset", "avg_snr_db", "avg_acc_msg0", "avg_acc_msg1", "num_samples"]
            )
        writer.writerow(
            [dataset, snr_val, float(test_avg_acc[0]), float(test_avg_acc[1]), num_samples]
        )


def write_robust_csv(results_dir, csv_path, dataset, sums):
    results_dir = Path(results_dir)
    csv_path = results_dir / csv_path
    file_exists = csv_path.exists()
    with open(csv_path, mode="a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(
                [
                    "dataset",
                    "attack_id",
                    "attack_name",
                    "avg_acc",
                    "avg_snr_total_db",
                    "avg_snr_vs_watermarked_db",
                    "batches_contributed",
                ]
            )
        for aid in WAVE_ATTACKS:
            nb = max(1, sums[aid]["batch_count"])
            writer.writerow(
                [
                    dataset,
                    aid,
                    ATTACK_NAMES.get(aid, "?"),
                    sums[aid]["acc_batch_sum"] / nb,
                    sums[aid]["snr_total_sum"] / nb,
                    sums[aid]["snr_vs_wm_sum"] / nb,
                    nb,
                ]
            )


def print_robust_results(sums):
    print("\n=== Per-attack results (average over batches) ===")
    header = "ID  Attack".ljust(32) + "Acc     SNR_total(dB)  SNR_vsWM(dB)"
    print(header)
    print("-" * len(header))
    for aid in WAVE_ATTACKS:
        nb = max(1, sums[aid]["batch_count"])
        avg_acc_batch = sums[aid]["acc_batch_sum"] / nb
        avg_snr_tot = sums[aid]["snr_total_sum"] / nb
        avg_snr_wm = sums[aid]["snr_vs_wm_sum"] / nb
        name = f"{aid}: {ATTACK_NAMES.get(aid,'?')}".ljust(32)
        print(f"{name}{avg_acc_batch:0.4f}   {avg_snr_tot:10.3f}     {avg_snr_wm:10.3f}")


D:\conda_envs\realtime-wm\lib\site-packages\librosa\util\files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [2]:
script_dir = r"D:\research\Realtime_WM\watermarking_model"

process_config, model_config, train_config = load_configs(script_dir)

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

CHECKPOINT_PATH = os.path.join(
    script_dir,
    "results/ckpt/pth/MSE_loudness_split_frequency_adaptive_soft_vad_phone_distortion_ep_60_2025-10-24_12_08_46.pth.tar",
)

In [3]:
encoder, decoder = load_encoder_decoder(
    process_config, model_config, train_config, CHECKPOINT_PATH, device
)

D:\research\Realtime_WM\watermarking_model\distortions\frequency.py:556: FutureWarning: Pass size=322 as keyword args. From version 0.10 passing these as positional arguments will result in an error
  fft_window = pad_center(fft_window, filter_length)
D:\research\Realtime_WM\watermarking_model\distortions\frequency.py:361: FutureWarning: Pass size=322 as keyword args. From version 0.10 passing these as positional arguments will result in an error
  fft_window = pad_center(fft_window, filter_length)
D:\research\Realtime_WM\watermarking_model\distortions\frequency.py:454: FutureWarning: Pass sr=16000, n_fft=322, n_mels=80, fmin=0.0, fmax=8000.0 as keyword args. From version 0.10 passing these as positional arguments will result in an error
  mel_basis = librosa_mel_fn(
